# 🚨 FNR Gap Monitoring Alert System
## Detecting Unequal Harm Across Operational Slices in Binary Classifiers

---

### 🧩 Problem Statement

**What problem are we solving?**

In deployed binary classifiers (e.g., medical diagnosis, fraud detection), labels become available weekly. Different operational slices (e.g., device type, hospital site) may experience different **False Negative Rates (FNR)**, which constitutes **unequal harm**.

**Why it matters:**
- A model might work well overall but miss more cases for certain groups
- Missing positive cases (false negatives) can cause serious harm (e.g., missed diagnoses)
- Regulatory and ethical requirements demand fair treatment across groups

**Real-world relevance:**
- Healthcare: Diagnostic AI missing more diseases at certain hospitals
- Finance: Fraud detection failing more on certain device types
- Insurance: Claim approval differing across regions

---

### 🪜 Steps to Solve the Problem

1. **Define the Metric**: FNR per slice and Gap calculation
2. **Specify Alert Threshold**: Gap > 10% triggers alert
3. **Set Time Window**: 4-week rolling window for stability
4. **Write Runbook**: Immediate actions when alert fires

**Key Formulas:**
```
FNR_g = FN_g / (TP_g + FN_g)
Gap = max_g(FNR_g) - min_g(FNR_g)
```

---

### 🎯 Expected Output (Overall)

- FNR calculated for each operational slice
- Gap metric showing disparity between worst and best slices
- Alert fired when Gap > 10%
- Runbook with actionable steps for response

---

## 📚 Section 1: Importing Required Libraries

Before we write any logic, we need to import the Python libraries that provide the functionality we need.

### 🔹 Line: `import numpy as np`

#### 2.1 What the line does
Imports the NumPy library and gives it the alias `np` for shorter references.

#### 2.2 Why it is used
NumPy provides efficient numerical operations, random number generation, and array manipulation.

#### 2.3 When to use it
Whenever you need mathematical operations, random sampling, or working with numerical arrays.

#### 2.4 Where to use it
- Data science projects
- Machine learning
- Scientific computing
- Statistical analysis

#### 2.5 How to use it (syntax + examples)
```python
import numpy as np
arr = np.array([1, 2, 3])
mean = np.mean(arr)
```

#### 2.6 How it works internally
Python's `import` statement loads the numpy module. The `as np` creates an alias.

#### 2.7 Output with sample examples
No direct output - this is a setup line.

In [ ]:
import numpy as np

### 🔹 Line: `import pandas as pd`

#### 2.1 What the line does
Imports the Pandas library with alias `pd`.

#### 2.2 Why it is used
Pandas provides DataFrame structures for tabular data manipulation.

#### 2.3 When to use it
When working with structured data (tables, CSVs, databases).

#### 2.4 Where to use it
- Data preprocessing
- Data analysis
- ETL pipelines

#### 2.5 How to use it
```python
import pandas as pd
df = pd.DataFrame({'col1': [1, 2], 'col2': [3, 4]})
```

#### 2.6 How it works internally
Loads pandas module and creates `pd` alias.

#### 2.7 Output
No direct output.

In [ ]:
import pandas as pd

### 🔹 Line: `from datetime import datetime, timedelta`

#### 2.1 What the line does
Imports `datetime` and `timedelta` classes from the datetime module.

#### 2.2 Why it is used
We need to work with dates for weekly data simulation.

#### 2.3 When to use it
- Date/time manipulation
- Time-series data
- Scheduling

#### 2.5 How to use it
```python
from datetime import datetime, timedelta
today = datetime.now()
next_week = today + timedelta(weeks=1)
```

In [ ]:
from datetime import datetime, timedelta
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

---

## 📚 Section 2: Configuration and Constants

Define the operational slices, alert threshold, and time window parameters.

### 🔹 Line: `OPERATIONAL_SLICES = ['Hospital_A', ...]`

#### 2.1 What the line does
Defines a list of operational slice names to monitor.

#### 2.2 Why it is used
We need to identify which groups (slices) to track for fairness monitoring.

#### 2.3 When to use it
When you have distinct subgroups in your deployed system.

#### 2.4 Where to use it
- Healthcare: Different hospitals, devices
- Finance: Different regions, products

**Real-life analogy:** Like tracking exam scores for different classrooms.

In [ ]:
# Define operational slices (groups to monitor)
OPERATIONAL_SLICES = ['Hospital_A', 'Hospital_B', 'Hospital_C', 'Device_Mobile', 'Device_Desktop']

# Alert configuration
ALERT_THRESHOLD = 0.10  # Alert if Gap > 10%
TIME_WINDOW_WEEKS = 4   # Rolling window of 4 weeks

# Seed for reproducibility
RANDOM_SEED = 42

print(f"Monitoring {len(OPERATIONAL_SLICES)} slices: {OPERATIONAL_SLICES}")
print(f"Alert Threshold: {ALERT_THRESHOLD*100}%")
print(f"Time Window: {TIME_WINDOW_WEEKS} weeks")

### 💼 Interview Perspective: Configuration

**Q: Why use constants instead of hard-coded values?**

A: Constants make code maintainable. Changing the threshold once at the top updates it everywhere.

**Q: How do you choose operational slices?**

A: Based on business context - groups where performance differences would indicate unfair treatment.

---

## 📚 Section 3: Data Simulation Function

Create a function to simulate weekly classification results across slices.

### ⚙️ Function: `simulate_weekly_classification_data`

**Purpose:** Generate synthetic weekly classification data for demonstration.

#### Parameter Explanations:

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `slices` | List[str] | - | Names of operational slices |
| `num_weeks` | int | 8 | Number of weeks to simulate |
| `base_fnr` | float | 0.15 | Base FNR around which slices vary |
| `fnr_variation` | float | 0.10 | Max variation between slices |
| `samples_per_slice` | int | 500 | Positive samples per slice/week |
| `seed` | int | 42 | Random seed for reproducibility |

In [ ]:
def simulate_weekly_classification_data(
    slices: List[str],
    num_weeks: int = 8,
    base_fnr: float = 0.15,
    fnr_variation: float = 0.10,
    samples_per_slice: int = 500,
    seed: int = RANDOM_SEED
) -> pd.DataFrame:
    """
    Simulate weekly classification results for multiple operational slices.
    
    Returns DataFrame with: week, slice, TP, FN, FNR
    """
    np.random.seed(seed)
    data = []
    start_date = datetime(2026, 1, 1)
    
    for week_num in range(num_weeks):
        week_date = start_date + timedelta(weeks=week_num)
        
        for slice_name in slices:
            # Assign different FNR to each slice
            slice_index = slices.index(slice_name)
            slice_fnr = base_fnr + (slice_index / len(slices)) * fnr_variation
            
            # Add random weekly variation
            weekly_fnr = slice_fnr + np.random.uniform(-0.02, 0.02)
            weekly_fnr = np.clip(weekly_fnr, 0.01, 0.50)
            
            # Calculate TP and FN
            total_positives = samples_per_slice
            fn = int(weekly_fnr * total_positives)
            tp = total_positives - fn
            
            data.append({
                'week': week_num + 1,
                'week_date': week_date.strftime('%Y-%m-%d'),
                'slice': slice_name,
                'TP': tp,
                'FN': fn,
                'total_positives': total_positives
            })
    
    return pd.DataFrame(data)

# Test the function
print("Function defined successfully!")

### 📌 Sample Example: Generate Data

In [ ]:
# Generate sample data
data = simulate_weekly_classification_data(
    slices=OPERATIONAL_SLICES,
    num_weeks=8,
    base_fnr=0.12,
    fnr_variation=0.15,
    samples_per_slice=500
)

print("📋 Simulated Data Preview:")
print(data.head(10).to_string(index=False))

---

## 📚 Section 4: FNR Calculation Functions

Implement the core FNR calculation: `FNR = FN / (TP + FN)`

### ⚙️ Function: `calculate_fnr_per_slice`

**Formula:** `FNR = FN / (TP + FN)`

**Real-life analogy:** If a security guard checks 100 bags and misses 15 dangerous items, FNR = 15/100 = 15%.

| Parameter | What it does | Why used |
|-----------|--------------|----------|
| `tp` | True Positives count | Cases correctly identified |
| `fn` | False Negatives count | Cases missed |

In [ ]:
def calculate_fnr_per_slice(tp: int, fn: int) -> float:
    """
    Calculate False Negative Rate for a single slice.
    
    Formula: FNR = FN / (TP + FN)
    
    Parameters:
    - tp: True Positives (correctly identified positives)
    - fn: False Negatives (missed positives)
    
    Returns:
    - FNR value between 0 and 1
    """
    total = tp + fn
    
    if total == 0:
        return 0.0  # Avoid division by zero
    
    fnr = fn / total
    return fnr

# Test with example
example_tp = 85
example_fn = 15
example_fnr = calculate_fnr_per_slice(example_tp, example_fn)
print(f"Example: TP={example_tp}, FN={example_fn}")
print(f"FNR = {example_fn} / ({example_tp} + {example_fn}) = {example_fnr:.4f} ({example_fnr*100:.2f}%)")

### ⚙️ Function: `calculate_fnr_for_all_slices`

Calculates FNR for all slices in the dataset.

In [ ]:
def calculate_fnr_for_all_slices(data: pd.DataFrame, week: int = None) -> Dict[str, float]:
    """
    Calculate FNR for all slices, optionally filtered by week.
    """
    if week is not None:
        filtered_data = data[data['week'] == week]
    else:
        filtered_data = data.groupby('slice').agg({'TP': 'sum', 'FN': 'sum'}).reset_index()
    
    fnr_dict = {}
    for _, row in filtered_data.iterrows():
        fnr = calculate_fnr_per_slice(row['TP'], row['FN'])
        fnr_dict[row['slice']] = fnr
    
    return fnr_dict

# Test: Calculate FNR for week 1
fnr_week1 = calculate_fnr_for_all_slices(data, week=1)
print("FNR per slice (Week 1):")
for slice_name, fnr in fnr_week1.items():
    print(f"  {slice_name}: {fnr:.4f} ({fnr*100:.2f}%)")

---

## 📚 Section 5: Gap Metric Calculation

**Formula:** `Gap = max_g(FNR_g) - min_g(FNR_g)`

This measures the disparity between the worst and best performing slices.

In [ ]:
def calculate_fnr_gap(fnr_dict: Dict[str, float]) -> Tuple[float, str, str]:
    """
    Calculate the FNR Gap metric.
    
    Formula: Gap = max_g(FNR_g) - min_g(FNR_g)
    
    Returns: (gap_value, worst_slice_name, best_slice_name)
    """
    if not fnr_dict:
        return 0.0, None, None
    
    max_fnr = max(fnr_dict.values())
    min_fnr = min(fnr_dict.values())
    
    worst_slice = max(fnr_dict, key=fnr_dict.get)
    best_slice = min(fnr_dict, key=fnr_dict.get)
    
    gap = max_fnr - min_fnr
    
    return gap, worst_slice, best_slice

# Test gap calculation
gap, worst, best = calculate_fnr_gap(fnr_week1)
print(f"\n📊 Gap Metric:")
print(f"  Gap = {gap:.4f} ({gap*100:.2f}%)")
print(f"  Worst: {worst} (FNR = {fnr_week1[worst]:.4f})")
print(f"  Best:  {best} (FNR = {fnr_week1[best]:.4f})")

### 💼 Interview Perspective: Gap Metric

**Q: Why use max-min instead of standard deviation?**

A: Max-min is simpler to explain and directly shows the worst-case disparity.

**Q: What's a good threshold?**

A: 10% is common in industry, but depends on risk tolerance and domain.

---

## 📚 Section 6: Rolling Window Aggregation

Aggregate data over a 4-week window for more stable metrics.

In [ ]:
def calculate_rolling_window_fnr(
    data: pd.DataFrame,
    current_week: int,
    window_weeks: int = TIME_WINDOW_WEEKS
) -> Dict[str, float]:
    """
    Calculate FNR per slice using a rolling time window.
    """
    start_week = max(1, current_week - window_weeks + 1)
    end_week = current_week
    
    # Filter data to window
    window_data = data[(data['week'] >= start_week) & (data['week'] <= end_week)]
    
    # Aggregate TP and FN per slice
    aggregated = window_data.groupby('slice').agg({'TP': 'sum', 'FN': 'sum'}).reset_index()
    
    # Calculate FNR for each slice
    fnr_dict = {}
    for _, row in aggregated.iterrows():
        fnr = calculate_fnr_per_slice(row['TP'], row['FN'])
        fnr_dict[row['slice']] = fnr
    
    return fnr_dict

# Test: Rolling window for week 6
fnr_rolling = calculate_rolling_window_fnr(data, current_week=6, window_weeks=4)
print(f"Rolling Window FNR (Weeks 3-6):")
for slice_name, fnr in sorted(fnr_rolling.items(), key=lambda x: x[1], reverse=True):
    print(f"  {slice_name}: {fnr:.4f} ({fnr*100:.2f}%)")

---

## 📚 Section 7: Alert Threshold Checking

In [ ]:
def check_alert_threshold(gap: float, threshold: float = ALERT_THRESHOLD) -> bool:
    """
    Check if the FNR gap exceeds the alert threshold.
    
    Returns: True if alert should fire
    """
    return gap > threshold

# Test alert check
gap_test, _, _ = calculate_fnr_gap(fnr_rolling)
alert = check_alert_threshold(gap_test)
print(f"Gap: {gap_test:.4f}, Threshold: {ALERT_THRESHOLD}")
print(f"Alert Status: {'🚨 ALERT FIRED' if alert else '✅ No Alert'}")

---

## 📚 Section 8: Runbook Implementation

The runbook defines immediate actions to take when an alert fires.

In [ ]:
def execute_runbook(gap: float, worst_slice: str, best_slice: str, fnr_dict: Dict[str, float], week: int) -> str:
    """
    Execute the runbook - immediate actions after alert fires.
    """
    runbook_log = f"""
{'#'*50}
# RUNBOOK EXECUTION - Week {week}
{'#'*50}

Alert Context:
  Gap: {gap:.4f} ({gap*100:.2f}%)
  Worst: {worst_slice} (FNR = {fnr_dict[worst_slice]:.4f})
  Best:  {best_slice} (FNR = {fnr_dict[best_slice]:.4f})

🔴 IMMEDIATE ACTIONS (24 hours):
1. Notify ML team lead
2. Check data quality for {worst_slice}
3. Compare feature distributions

🟡 SHORT-TERM ACTIONS (1 week):
4. Root cause analysis
5. Document findings
6. Propose mitigation

🟢 FOLLOW-UP ACTIONS (1 month):
7. Implement fix
8. Verify resolution
9. Update processes
{'#'*50}
"""
    return runbook_log

# Test runbook
if alert:
    gap_val, worst, best = calculate_fnr_gap(fnr_rolling)
    runbook = execute_runbook(gap_val, worst, best, fnr_rolling, 6)
    print(runbook)

---

## 📚 Section 9: Complete Monitoring Pipeline

Run the full monitoring system across all weeks.

In [ ]:
def run_fnr_gap_monitoring(data: pd.DataFrame, threshold: float = ALERT_THRESHOLD, 
                           window_weeks: int = TIME_WINDOW_WEEKS) -> pd.DataFrame:
    """
    Run the complete FNR Gap monitoring pipeline.
    """
    max_week = data['week'].max()
    results = []
    
    print("="*50)
    print("🔍 FNR GAP MONITORING SYSTEM")
    print("="*50)
    
    for week in range(window_weeks, max_week + 1):
        fnr_dict = calculate_rolling_window_fnr(data, week, window_weeks)
        gap, worst_slice, best_slice = calculate_fnr_gap(fnr_dict)
        alert_fired = check_alert_threshold(gap, threshold)
        
        status = "🚨 ALERT" if alert_fired else "✅ OK"
        print(f"Week {week}: Gap = {gap:.4f} ({gap*100:.1f}%) {status}")
        
        results.append({
            'week': week,
            'gap': gap,
            'worst_slice': worst_slice,
            'best_slice': best_slice,
            'alert_fired': alert_fired
        })
    
    print("="*50)
    return pd.DataFrame(results)

# Run monitoring
results = run_fnr_gap_monitoring(data)

---

## 📚 Section 10: Summary and Interview Preparation

### Key Takeaways

1. **FNR Gap** measures disparity across slices
2. **10% threshold** is industry standard
3. **4-week rolling window** reduces noise
4. **Runbook** provides clear response steps

### Interview Questions

| Question | Answer |
|----------|--------|
| What is FNR? | FN / (TP + FN) - proportion of missed positives |
| Why monitor slices? | Overall metrics hide group disparities |
| What triggers an alert? | Gap > threshold (e.g., 10%) |
| Why use rolling window? | Smooths weekly noise |

In [ ]:
# Final summary
print("\n📊 MONITORING SUMMARY")
print("="*40)
print(f"Total Weeks: {len(results)}")
print(f"Alerts Fired: {results['alert_fired'].sum()}")
print(f"Alert Rate: {results['alert_fired'].mean()*100:.1f}%")
print("\n✅ Monitoring Complete!")